# VGG Feature Visualization

This notebook loads the project's pretrained VGG encoder and displays intermediate feature activations for the bundled example images. Run it from the repository root.

In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

PROJECT_ROOT = Path.cwd().resolve()
NST_CODE_DIR = PROJECT_ROOT / 'NST_Code'
if not (NST_CODE_DIR / 'app.py').is_file():
    raise FileNotFoundError('Open this notebook with the repository root as the working directory.')
sys.path.insert(0, str(NST_CODE_DIR))
from utils.models import VGGEncoder

EXAMPLES_DIR = NST_CODE_DIR / 'examples'
VGG_WEIGHTS = NST_CODE_DIR / 'vgg_normalised.pth'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
image_size = 512
transform = transforms.Compose([transforms.Resize((image_size, image_size)), transforms.ToTensor()])
encoder = VGGEncoder(VGG_WEIGHTS, map_location=device).to(device).eval()
print('Using device:', device)

In [ ]:
image_paths = {
    'Original Brad Pitt': EXAMPLES_DIR / 'brad_pitt.jpg',
    'Sketch style result': EXAMPLES_DIR / 'stylized_brad_pitt.jpg',
    'Picasso style result': EXAMPLES_DIR / 'stylized_brad_pitt (1).jpg',
}
missing = [str(path) for path in image_paths.values() if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing example images: ' + ', '.join(missing))

def load_image(path):
    with Image.open(path) as source:
        image = source.convert('RGB')
    tensor = transform(image).unsqueeze(0).to(device)
    return image, tensor

def extract_features(tensor):
    with torch.inference_mode():
        h1, h2, h3, h4 = encoder(tensor)
    return {'relu1_1': h1, 'relu2_1': h2, 'relu3_1': h3, 'relu4_1': h4}

def activation_map(feature_tensor):
    activation = feature_tensor[0].mean(dim=0).detach().cpu()
    activation = activation - activation.min()
    return activation / (activation.max() + 1e-8)

loaded_images = {}
feature_bank = {}
for name, path in image_paths.items():
    image, tensor = load_image(path)
    loaded_images[name] = image
    feature_bank[name] = extract_features(tensor)
    print(name, {layer: tuple(value.shape) for layer, value in feature_bank[name].items()})

In [ ]:
layers_to_show = ['relu1_1', 'relu2_1', 'relu3_1', 'relu4_1']
fig, axes = plt.subplots(len(layers_to_show) + 1, len(image_paths), figsize=(5 * len(image_paths), 3.5 * (len(layers_to_show) + 1)))
for column, name in enumerate(image_paths):
    axes[0, column].imshow(loaded_images[name])
    axes[0, column].set_title(name)
    axes[0, column].axis('off')
    for row, layer in enumerate(layers_to_show, start=1):
        axes[row, column].imshow(activation_map(feature_bank[name][layer]), cmap='viridis')
        axes[row, column].axis('off')
for row, label in enumerate(['Input'] + layers_to_show):
    axes[row, 0].set_ylabel(label, rotation=90, labelpad=12)
plt.suptitle('VGG content features for original and stylized images')
plt.tight_layout()
plt.show()